# 06 — Convolutional Networks from Scratch

**CS231n Lecture 5.** The fully-connected net in notebook 05 works, but it is
structurally wrong for images in three ways:

1. **It has no idea pixels have neighbours.** Shuffle every pixel with a fixed
   permutation and it trains to the same accuracy. That is a damning fact.
2. **It has no translation equivariance.** A cat in the top-left and the same
   cat in the bottom-right hit completely different weights.
3. **It is enormous.** One fully-connected layer from 200×200×3 to 1000 units is
   120 million parameters.

Convolution fixes all three with two ideas: **local connectivity** and **weight
sharing**.

| You write | |
|---|---|
| `conv_output_size` | the formula you must know cold |
| `conv_forward_naive` | the honest 4-deep loop |
| `conv_backward_naive` | three gradients, one of which people always get wrong |
| `max_pool_forward_naive` / `_backward` | the max gate, spatially |
| `im2col` | convolution as a single matrix multiply |

Layout is **NCHW** throughout, matching PyTorch:
`x (N, C, H, W)`, `w (F, C, HH, WW)`, `out (N, F, H', W')`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cs231n_utils import (get_cifar10, CLASSES, rel_error, grade, show_solution,
                          use, show_images, eval_numerical_gradient_array,
                          setup_plots, SERIES)

setup_plots()
np.random.seed(0)

## 1. The output size formula

$$ H_{out} = \frac{H + 2P - HH}{S} + 1 $$

where `H` is input height, `HH` the filter height, `P` the padding and `S` the
stride. Same for width.

**When this is not an integer, the filter does not tile the input exactly** —
the last window would hang off the edge. PyTorch and friends take the **floor**
and silently drop the leftover row/column rather than raising.

That silence cuts both ways. `3x3, stride 2, pad 1` gives `16.5` on a 32-pixel
input, floors to 16, and is the standard downsampling block in every ResNet —
completely intentional. But the same silence means a genuinely wrong
configuration also just... runs, quietly discarding pixels down one edge. Always
compute the number yourself and decide whether you meant it.

The combination worth memorising is **`stride 1`, `filter F`, `pad (F-1)/2`**,
which preserves the spatial size exactly:

| Filter | Pad for "same" |
|---|---|
| 3×3 | 1 |
| 5×5 | 2 |
| 7×7 | 3 |

Preserving size matters more than it looks. Without padding, a 3×3 conv shrinks
the image by 2 pixels per layer — ten layers into a 32×32 image and you have
12×12 left, having thrown away the border repeatedly.

### TODO 1 — the formula

Return floats (not ints) so a non-integer result is visible rather than
silently floored.

In [ ]:
def conv_output_size(H, W, HH, WW, stride, pad):
    """-> (H_out, W_out) as floats."""
    pass  # TODO

In [ ]:
grade('nb06', 'conv_output_size', conv_output_size, verbose=False)

In [ ]:
cos = use('nb06', 'conv_output_size', conv_output_size)

print("%-42s %s" % ("configuration", "output"))
for desc, args in [
    ("32x32, 5x5 filter, stride 1, pad 2  (same)", (32, 32, 5, 5, 1, 2)),
    ("32x32, 3x3 filter, stride 1, pad 1  (same)", (32, 32, 3, 3, 1, 1)),
    ("32x32, 3x3 filter, stride 1, pad 0", (32, 32, 3, 3, 1, 0)),
    ("32x32, 3x3, stride 2, pad 1  (ResNet halve)", (32, 32, 3, 3, 2, 1)),
    ("227x227, 11x11, stride 4, pad 0  (AlexNet)", (227, 227, 11, 11, 4, 0)),
    ("7x7, 2x2 filter, stride 2, pad 0", (7, 7, 2, 2, 2, 0)),
]:
    h, w = cos(*args)
    exact = (h == int(h) and w == int(w))
    flag = "" if exact else "   -> floors to %d x %d, edge dropped" % (int(h), int(w))
    print("%-44s %5.1f x %-5.1f%s" % (desc, h, w, flag))

### Parameter count — where convolution actually wins

A conv layer's parameter count is `F * (C * HH * WW) + F`. Note what is
**absent**: the input's height and width. The same filter slides everywhere, so
the parameter count is independent of image size.

In [ ]:
H = W = 200
C, F = 3, 64
fc_params = (C * H * W) * 1000 + 1000
conv_params = F * (C * 3 * 3) + F
print("fully-connected 200x200x3 -> 1000 units : {:>12,} parameters".format(fc_params))
print("conv 3x3, 3 -> 64 channels              : {:>12,} parameters".format(conv_params))
print("ratio: %.0f x fewer" % (fc_params / conv_params))
print("\nand the conv layer works on ANY input size; the FC layer does not.")

## 2. TODO 2 — convolution forward

Four nested loops: example, filter, output row, output column. Slow and
obviously correct — write this version first, always.

Two things to get right:

- **Pad only the spatial axes.** `np.pad(x, ((0,0), (0,0), (pad,pad), (pad,pad)))`.
  Padding N or C is a classic bug that produces plausible-looking garbage.
- **Each filter spans all input channels.** `w[f]` has shape `(C, HH, WW)` and
  the window has shape `(C, HH, WW)`; one output value is the sum over *all
  three* axes. A conv filter is 3D, not 2D. This is the detail most people get
  wrong when first writing it.

In [ ]:
def conv_forward_naive(x, w, b, conv_param):
    """x (N,C,H,W), w (F,C,HH,WW), b (F,), conv_param has 'stride' and 'pad'.

    -> (out (N,F,H_out,W_out), cache)
    """
    pass  # TODO

In [ ]:
grade('nb06', 'conv_forward_naive', conv_forward_naive, verbose=False)

### Check it against something you can verify by eye

Rather than trust a reference, run known image-processing kernels through your
convolution and confirm they do what those kernels are supposed to do. If your
Sobel filter produces edges, your indexing is right.

In [ ]:
cf = use('nb06', 'conv_forward_naive', conv_forward_naive)

data = get_cifar10(num_train=200, num_val=50, num_test=50, flatten=False,
                   subtract_mean=False, verbose=False)
img = data['X_train'][3:4] / 255.0          # (1, 3, 32, 32)

kernels = {
    'identity':   np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float),
    'blur':       np.ones((3, 3)) / 9.0,
    'sharpen':    np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
    'sobel x':    np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float),
    'sobel y':    np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float),
}

# one filter per kernel, applied identically to each of the 3 channels
F = len(kernels)
w = np.zeros((F, 3, 3, 3))
for i, k in enumerate(kernels.values()):
    for c in range(3):
        w[i, c] = k / 3.0
b = np.zeros(F)

out, _ = cf(img, w, b, {'stride': 1, 'pad': 1})
print("in", img.shape, "-> out", out.shape)

panels = [img[0].transpose(1, 2, 0)] + [out[0, i] for i in range(F)]
names = ['original'] + list(kernels.keys())
fig, axes = plt.subplots(1, F + 1, figsize=(2.0 * (F + 1), 2.3))
for ax, p, nm in zip(axes, panels, names):
    p = np.asarray(p, dtype=float)
    ax.imshow(np.clip((p - p.min()) / max(np.ptp(p), 1e-9), 0, 1),
              cmap=None if p.ndim == 3 else 'gray')
    ax.set_title(nm, fontsize=8, color='#52514e')
    ax.axis('off')
plt.show()

The identity gives you the image back, blur blurs, and the two Sobels light up
vertical and horizontal edges respectively. That is a far stronger check than
matching a reference, because it would catch a transposed index that happens to
be self-consistent.

> Recognise those kernels? They are the ones you hand-designed in your
> classical-CV notebooks. **The only difference now is that the kernel values
> are learned by gradient descent instead of chosen by a person.** That is
> genuinely the whole idea of a convnet.

## 3. TODO 3 — convolution backward

Three gradients. Two are easy; one has a trap.

- `db[f]` — sum of `dout[:, f]` over everything. The bias is added to every
  spatial position, so every position's gradient comes back to it.
- `dw[f]` — for each output position, the multiply gate says `dw[f] +=
  window * dout[n,f,i,j]`.
- `dx` — **the trap.** With stride < filter size, receptive fields *overlap*, so
  one input pixel feeds many output positions. That is the branching case from
  notebook 04: you must **accumulate with `+=`**, never assign. Then strip the
  padding at the end so `dx` matches `x`'s shape.

In [ ]:
def conv_backward_naive(dout, cache):
    """-> (dx, dw, db)."""
    pass  # TODO

In [ ]:
grade('nb06', 'conv_backward_naive', conv_backward_naive, verbose=False)

In [ ]:
cb = use('nb06', 'conv_backward_naive', conv_backward_naive)

rng = np.random.RandomState(0)
xc = rng.randn(2, 3, 6, 6)
wc, bc = rng.randn(4, 3, 3, 3), rng.randn(4)
cp = {'stride': 1, 'pad': 1}
oc, cache_c = cf(xc, wc, bc, cp)
dc = rng.randn(*oc.shape)
dx, dw, db = cb(dc, cache_c)

print("stride 1, pad 1:")
print("  dx %.2e" % rel_error(dx, eval_numerical_gradient_array(
    lambda t: cf(t, wc, bc, cp)[0], xc.copy(), dc)))
print("  dw %.2e" % rel_error(dw, eval_numerical_gradient_array(
    lambda t: cf(xc, t, bc, cp)[0], wc.copy(), dc)))
print("  db %.2e" % rel_error(db, eval_numerical_gradient_array(
    lambda t: cf(xc, wc, t, cp)[0], bc.copy(), dc)))

# stride 2 exercises the padding-strip and accumulation paths differently
cp2 = {'stride': 2, 'pad': 1}
o2, c2 = cf(xc, wc, bc, cp2)
d2 = rng.randn(*o2.shape)
dx2, dw2, db2 = cb(d2, c2)
print("stride 2, pad 1:")
print("  dx %.2e" % rel_error(dx2, eval_numerical_gradient_array(
    lambda t: cf(t, wc, bc, cp2)[0], xc.copy(), d2)))
print("  dw %.2e" % rel_error(dw2, eval_numerical_gradient_array(
    lambda t: cf(xc, t, bc, cp2)[0], wc.copy(), d2)))

## 4. TODO 4 — max pooling

Pooling downsamples each channel independently. No parameters, no
cross-channel mixing — just "take the strongest response in each
neighbourhood".

Backward is the **max gate** from notebook 04, applied per window: the winner
gets all of `dout`, everyone else gets zero. Non-overlapping pooling (the usual
2×2 stride 2) means no accumulation is needed, but use `+=` anyway so
overlapping pooling also works.

In [ ]:
def max_pool_forward_naive(x, pool_param):
    """pool_param has 'pool_height', 'pool_width', 'stride'. -> (out, cache)"""
    pass  # TODO

def max_pool_backward_naive(dout, cache):
    """-> dx"""
    pass  # TODO

In [ ]:
grade('nb06', 'max_pool_forward_naive', max_pool_forward_naive, verbose=False)
grade('nb06', 'max_pool_backward_naive', max_pool_backward_naive, verbose=False)

In [ ]:
pf = use('nb06', 'max_pool_forward_naive', max_pool_forward_naive)
pb = use('nb06', 'max_pool_backward_naive', max_pool_backward_naive)

xp = rng.randn(2, 3, 6, 6)
pp = {'pool_height': 2, 'pool_width': 2, 'stride': 2}
op, cachep = pf(xp, pp)
dp = rng.randn(*op.shape)
print("pool %s -> %s" % (xp.shape, op.shape))
print("dx rel error %.2e" % rel_error(pb(dp, cachep),
      eval_numerical_gradient_array(lambda t: pf(t, pp)[0], xp.copy(), dp)))

In [ ]:
# The gradient is sparse: only 1 in 4 positions gets anything back.
dxp = pb(np.ones_like(dp), cachep)
print("fraction of input positions receiving gradient: %.2f (expect 0.25)"
      % np.mean(dxp != 0))

> **Pooling is falling out of fashion.** Strided convolution does the same
> downsampling while *learning* how to combine the window instead of blindly
> taking the max, and it keeps the network fully differentiable everywhere.
> ResNet uses one max-pool at the very start and none after; many modern
> architectures use none at all. Know it because you will read it in older
> papers and because the max-gate gradient is instructive.

## 5. Making it fast — `im2col`

Your naive conv is unusably slow: four nested python loops with a numpy call in
the innermost. The standard fix is not a better loop, it is a **change of
representation**, and it is the same lesson as notebook 01.

**The idea:** flatten every receptive field into a column of a big matrix. If
each patch is `C*HH*WW` values and there are `N*H_out*W_out` patches, you get

```
cols : (C*HH*WW, N*H_out*W_out)
w    : (F, C*HH*WW)          after reshaping
out  = w @ cols + b          ->  (F, N*H_out*W_out)   then reshape
```

**Convolution becomes one matrix multiply.** This is what every framework
actually does on CPU, and it is why BLAS quality determines convnet speed.

The cost is memory: overlapping patches are duplicated, so `cols` is roughly
`HH*WW/stride²` times larger than the input. For 3×3 stride 1, that is 9×. A
completely standard time/memory trade.

### TODO 5 — `im2col`

`im2col_indices` (provided) precomputes the fancy-index arrays. Your job is to
pad, gather, and reshape into `(C*HH*WW, N*H_out*W_out)`.

In [ ]:
from solutions.nb06 import im2col_indices, col2im

def im2col(x, HH, WW, stride, pad):
    """-> (C*HH*WW, N*H_out*W_out)

    Steps:
      1. pad the spatial axes of x
      2. k, i, j = im2col_indices(x.shape, HH, WW, stride, pad)
      3. cols = xp[:, k, i, j]        -> (N, C*HH*WW, H_out*W_out)
      4. transpose to (C*HH*WW, H_out*W_out, N) and reshape to 2D
    """
    pass  # TODO

In [ ]:
grade('nb06', 'im2col', im2col, verbose=False)

In [ ]:
i2c = use('nb06', 'im2col', im2col)

xs = rng.randn(2, 3, 5, 5)
cols = i2c(xs, 3, 3, 1, 1)
print("x %s -> cols %s" % (xs.shape, cols.shape))
print("  expected (C*HH*WW, N*H_out*W_out) = (%d, %d)" % (3 * 3 * 3, 2 * 5 * 5))
print("memory blow-up: %.1fx" % (cols.size / xs.size))

### TODO 6 — convolution as a matmul

In [ ]:
def conv_forward_im2col(x, w, b, conv_param):
    """Same output as conv_forward_naive, computed as one matrix multiply.

    -> (out, cache).  Return cache = (x, w, b, conv_param, cols).

    Careful with the final reshape: w.reshape(F,-1) @ cols is
    (F, N*H_out*W_out). The column order from im2col is (F, H_out, W_out, N),
    so reshape to that and then transpose(3, 0, 1, 2).
    """
    pass  # TODO

In [ ]:
grade('nb06', 'conv_forward_im2col', conv_forward_im2col, verbose=False)

In [ ]:
cfi = use('nb06', 'conv_forward_im2col', conv_forward_im2col)

xb = np.random.randn(8, 3, 32, 32)
wb, bb = np.random.randn(16, 3, 3, 3), np.random.randn(16)
cpb = {'stride': 1, 'pad': 1}

t0 = time.time(); o_naive, _ = cf(xb, wb, bb, cpb);  t_naive = time.time() - t0
t0 = time.time(); o_fast,  _ = cfi(xb, wb, bb, cpb); t_fast  = time.time() - t0

print("naive   %.3f s" % t_naive)
print("im2col  %.4f s" % t_fast)
print("speedup %.0fx" % (t_naive / max(t_fast, 1e-9)))
print("same answer: rel error %.2e" % rel_error(o_naive, o_fast))

### `col2im` and why the backward pass needs it

The backward pass has to scatter gradients from columns back to pixels. Because
`im2col` **duplicated** overlapping pixels, `col2im` must **add** the
contributions — `np.add.at`, not assignment. Same branching rule, third time.

In [ ]:
xr = rng.randn(2, 3, 5, 5)
cols_r = i2c(xr, 3, 3, 1, 1)
counts = col2im(np.ones_like(cols_r), xr.shape, 3, 3, 1, 1)
restored = col2im(cols_r, xr.shape, 3, 3, 1, 1)

print("each pixel was duplicated this many times (centre 3x3 block):")
print(counts[0, 0].astype(int))
print("\ncol2im(im2col(x)) / counts == x :",
      np.allclose(restored / counts, xr))

The count map shows exactly why: a corner pixel belongs to 4 receptive fields,
an edge pixel to 6, an interior pixel to 9. Dividing by the counts recovers the
original — but in the *gradient* pass you keep the sum, because each of those
receptive fields genuinely contributed.

## 6. What convolution buys you, measured

In [ ]:
# Translation equivariance: shift the input, the feature map shifts with it.
base = np.zeros((1, 1, 16, 16))
base[0, 0, 6:10, 6:10] = 1.0
shifted = np.roll(base, 3, axis=3)

edge = np.array([[[[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]]])
bz = np.zeros(1)
f_base, _ = cf(base, edge, bz, {'stride': 1, 'pad': 1})
f_shift, _ = cf(shifted, edge, bz, {'stride': 1, 'pad': 1})

print("conv(shift(x)) == shift(conv(x)) :",
      np.allclose(np.roll(f_base, 3, axis=3), f_shift))
print("  -> this is translation EQUIVARIANCE, and an FC layer has none of it.")

fig, axes = plt.subplots(1, 4, figsize=(8.4, 2.3))
for ax, im, nm in zip(axes,
                      [base[0, 0], f_base[0, 0], shifted[0, 0], f_shift[0, 0]],
                      ['input', 'conv(input)', 'shifted', 'conv(shifted)']):
    ax.imshow(im, cmap='gray'); ax.set_title(nm, fontsize=8, color='#52514e')
    ax.axis('off')
plt.show()

## 7. Recap

1. Write the output-size formula. What does a non-integer result mean?
2. Give the "same" padding for 3×3, 5×5 and 7×7 at stride 1.
3. A conv layer's parameter count does not involve `H` or `W`. Why, and why
   does that matter?
4. Why is a conv filter 3-dimensional rather than 2-dimensional?
5. In `conv_backward_naive`, why must `dx` be accumulated rather than assigned?
6. What fraction of positions receive gradient through 2×2 stride-2 max pool?
7. What does `im2col` trade away to get its speed, and by roughly what factor
   for a 3×3 stride-1 conv?
8. Why must `col2im` use `np.add.at` rather than indexed assignment?
9. State the difference between translation *equivariance* and *invariance*.
   Which does convolution give you, and which does pooling contribute to?

In [ ]:
# show_solution('nb06', 'conv_forward_naive')
# show_solution('nb06', 'conv_backward_naive')
# show_solution('nb06', 'im2col')